In [ ]:
import requests
import os
from lxml import etree


OC = "cleftin"   # 국가법령정보센터 ID

BASE_URL = "https://www.law.go.kr/DRF"


# -----------------------------
# 1. 법령 검색
# -----------------------------
def search_law(query):

    url = f"{BASE_URL}/lawSearch.do"

    params = {
        "OC": OC,
        "target": "law",
        "type": "XML",
        "query": query
    }

    res = requests.get(url, params=params)
    res.raise_for_status()

    return res.text

# -----------------------------
# 1-2. 행정규칙 검색(접구선통 기술기준)
# -----------------------------
def search_adm_rule(query):

    url = f"{BASE_URL}/lawSearch.do"

    params = {
        "OC": OC,
        "target": "admrul",
        # "ID": "2100000268620",   
        "type": "XML",
        "query": query
    }

    res = requests.get(url, params=params)
    res.raise_for_status()

    print(res.text[:500])  # 디버깅

    return res.text

# -----------------------------
# 2. 법령/행정규칙 ID 추출 (간단 파싱)
# -----------------------------
def extract_admrule_id(xml_text):

    root = etree.fromstring(xml_text.encode("utf-8"))

    # 🔥 핵심: 일련번호 사용
    ids = root.xpath("//행정규칙일련번호/text()")

    if not ids:
        print(xml_text[:1000])
        raise Exception("행정규칙일련번호 없음")

    return ids[0]


def extract_law_id(xml_text):

    start = xml_text.find("<법령ID>")
    end = xml_text.find("</법령ID>")

    if start == -1:
        raise Exception("법령ID 찾기 실패")

    return xml_text[start+6:end]


# -----------------------------
# 3. 법령 본문 다운로드
# -----------------------------
def download_law(law_id, save_type="XML"):

    url = f"{BASE_URL}/lawService.do"

    params = {
        "OC": OC,
        "target": "law",
        "type": save_type,  # XML / HTML
        "ID": law_id
    }

    res = requests.get(url, params=params)
    res.raise_for_status()

    return res.text

# -----------------------------
# 3-2. 행정규칙(고시) 본문 다운로드
# -----------------------------
def download_adm_rule(rule_id):

    url = "https://www.law.go.kr/DRF/lawService.do"

    params = {
        "OC": OC,
        "target": "admrul",
        "type": "XML",
        "ID": rule_id   # 🔥 2100000268620
    }

    res = requests.get(url, params=params)

    print("status:", res.status_code)

    res.raise_for_status()

    return res.text


# -----------------------------
# 4. 파일 저장
# -----------------------------
def save_file(content, filename):

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"파일 저장 완료: {filename}")

def debug_response(res):

    print("status:", res.status_code)
    print("length:", len(res.text))

    if not res.text.strip():
        print("❌ 응답이 비어 있음")
        print("👉 OC 또는 query 문제")
        return False

    print(res.text[:500])
    return True


# -----------------------------
# 5. 법률 검색 전체 실행
# -----------------------------
def main_law():

    query = "정보통신공사업법"

    print("1. 법령 검색 중...")
    search_result = search_law(query)

    law_id = extract_law_id(search_result)
    print(f"법령ID: {law_id}")

    print("2. 법령 다운로드 중...")
    law_xml = download_law(law_id, save_type="XML")

    filename = f"{query}.xml"
    save_file(law_xml, filename)

    print("\n--- 일부 출력 ---")
    print(law_xml[:1000])  # 앞부분 출력

# -----------------------------
# 5-2. 행정규칙(고시) 검색 전체 실행
# -----------------------------
def main_admrule():

    query = "접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준"

    print("1. 행정규칙 검색 중...")
    search_result = search_adm_rule(query)

    rule_id = extract_admrule_id(search_result)  
    print(f"행정규칙ID: {rule_id}")

    print("2. 행정규칙 다운로드 중...")
    rule_xml = download_adm_rule(rule_id)

    filename = "접구선통기술기준.xml"
    save_file(rule_xml, filename)

    print("\n--- 일부 출력 ---")
    print(rule_xml[:1000])

if __name__ == "__main__":
    main_admrule()

1. 행정규칙 검색 중...
<?xml version="1.0" encoding="UTF-8"?><AdmRulSearch><target>admrul</target><키워드>접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준</키워드><section>admRulNm</section><totalCnt>1</totalCnt><page>1</page><numOfRows>1</numOfRows><resultCode>00</resultCode><resultMsg>success</resultMsg><admrul id="1"><행정규칙일련번호>2100000268620</행정규칙일련번호><행정규칙명><![CDATA[접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준]]></행정규칙명><행정규칙종류>고시</행정규칙종류><발령일자>20251128</발령일자><발령번호>2025-10</발령번호><소관부처명>국립전파연구원</소관부처명><현행연혁구분>현행</현행연혁구분><제개정구분코드>200403</제개정구분코드>
행정규칙ID: 2100000268620
2. 행정규칙 다운로드 중...
status: 200
파일 저장 완료: 통신기술기준.xml

--- 일부 출력 ---
<?xml version="1.0" encoding="UTF-8"?><AdmRulService><행정규칙기본정보><행정규칙일련번호>2100000268620</행정규칙일련번호><행정규칙명><![CDATA[접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준]]></행정규칙명><행정규칙종류>고시</행정규칙종류><행정규칙종류코드>B0003</행정규칙종류코드><발령일자>20251128</발령일자><발령번호>2025-10</발령번호><제개정구분명>일부개정</제개정구분명><제개정구분코드>200403</제개정구분코드><조문형식여부>Y</조문형식여부><행정규칙ID>49823</행정규칙ID><소관부처명>국립전파연구원</소관부처명><소관부처코드>1721000</소관부처코드><상위부처명>과학기술정보통신부</상

In [26]:
import requests
from lxml import etree

OC = "cleftin"
BASE_URL = "https://www.law.go.kr/DRF"


def request_api(endpoint, params):

    url = f"{BASE_URL}/{endpoint}"

    res = requests.get(url, params=params)

    print(f"[DEBUG] {url}")
    print(f"[DEBUG] status: {res.status_code}")

    res.raise_for_status()

    if not res.text.strip():
        raise Exception("❌ API 응답이 비어 있음")

    return res.text


def save_file(content, filename):

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"✔ 저장 완료: {filename}")


# -----------------------------
# 법령 검색
# -----------------------------
def search_law(query):

    params = {
        "OC": OC,
        "target": "law",
        "type": "XML",
        "query": query
    }

    return request_api("lawSearch.do", params)


# -----------------------------
# 법령 ID 추출
# -----------------------------
def extract_law_id(xml_text):

    root = etree.fromstring(xml_text.encode("utf-8"))

    ids = root.xpath("//법령ID/text()")

    if not ids:
        raise Exception("❌ 법령ID 없음")

    return ids[0]


# -----------------------------
# 법령 다운로드
# -----------------------------
def download_law(law_id, save_type="XML"):

    params = {
        "OC": OC,
        "target": "law",
        "type": save_type,
        "ID": law_id
    }

    return request_api("lawService.do", params)


# -----------------------------
# 실행
# -----------------------------
def run_law():

    query = "방송통신설비의 기술기준에 관한 규정"

    print("\n=== 법령 처리 시작 ===")

    xml = search_law(query)
    law_id = extract_law_id(xml)

    print(f"법령ID: {law_id}")

    law_xml = download_law(law_id)

    save_file(law_xml, f"{query}.xml")

    print(law_xml[:500])


# -----------------------------
# 행정규칙 검색
# -----------------------------
def search_admrule(query):

    params = {
        "OC": OC,
        "target": "admrul",
        "type": "XML",
        "query": query
    }

    return request_api("lawSearch.do", params)


# -----------------------------
# 행정규칙 ID 추출 (일련번호 사용)
# -----------------------------
def extract_admrule_id(xml_text):

    root = etree.fromstring(xml_text.encode("utf-8"))

    ids = root.xpath("//행정규칙일련번호/text()")

    if not ids:
        print(xml_text[:1000])
        raise Exception("❌ 행정규칙일련번호 없음")

    return ids[0]


# -----------------------------
# 행정규칙 다운로드
# -----------------------------
def download_admrule(rule_id):

    params = {
        "OC": OC,
        "target": "admrul",
        "type": "XML",
        "ID": rule_id
    }

    return request_api("lawService.do", params)


# -----------------------------
# 실행
# -----------------------------
def run_admrule():

    query = "접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준"

    print("\n=== 행정규칙 처리 시작 ===")

    xml = search_admrule(query)
    rule_id = extract_admrule_id(xml)

    print(f"행정규칙 일련번호: {rule_id}")

    rule_xml = download_admrule(rule_id)

    # save_file(rule_xml, "접구선통기술기준.xml")
    save_file(rule_xml, f"{query}.xml")

    print(rule_xml[:500])


if __name__ == "__main__":

    mode = "admrule"   # "law" or "admrule"

    if mode == "law":
        run_law()

    elif mode == "admrule":
        run_admrule()

    else:
        print("❌ mode 오류")


=== 행정규칙 처리 시작 ===
[DEBUG] https://www.law.go.kr/DRF/lawSearch.do
[DEBUG] status: 200
행정규칙 일련번호: 2100000268620
[DEBUG] https://www.law.go.kr/DRF/lawService.do
[DEBUG] status: 200
✔ 저장 완료: 접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준.xml
<?xml version="1.0" encoding="UTF-8"?><AdmRulService><행정규칙기본정보><행정규칙일련번호>2100000268620</행정규칙일련번호><행정규칙명><![CDATA[접지설비·구내통신설비·선로설비 및 통신공동구등에 대한 기술기준]]></행정규칙명><행정규칙종류>고시</행정규칙종류><행정규칙종류코드>B0003</행정규칙종류코드><발령일자>20251128</발령일자><발령번호>2025-10</발령번호><제개정구분명>일부개정</제개정구분명><제개정구분코드>200403</제개정구분코드><조문형식여부>Y</조문형식여부><행정규칙ID>49823</행정규칙ID><소관부처명>국립전파연구원</소관부처명><소관부처코드>1721000</소관부처코드><상위부처명>과학기술정보통신부</상위부처명><담당부서기관코드>1721141</담당부서기관코드><담당부서기관명>국립전파연구원(기술기준과)</담당부서기관명><담당자명>임병철</담당자명><전화번호>061-338-4611</전화번
